# ZaDeep Fashion — Colab Quickstart
Run all cells top to bottom. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

In [ ]:
# 1. Clone the repo
!git clone https://github.com/nouryehia/zadeep-fashion.git
%cd zadeep-fashion

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Download the Kaggle Fashion Product Images dataset
# You need a Kaggle API key. Upload kaggle.json here first.
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paramaggarwal/fashion-product-images-small -p data/ --unzip

In [ ]:
# 4. Build the FAISS index (uses GPU automatically)
# --max_items 5000 for a quick test; remove it for the full 44K catalog
!python scripts/build_index.py \
    --csv data/fashion-dataset/styles.csv \
    --images data/fashion-dataset/images \
    --output models/index \
    --batch_size 64 \
    --max_items 5000

In [ ]:
# 5. Quick search test (no UI)
import sys; sys.path.insert(0, '.')
from src.search.search_engine import FashionSearchEngine
from PIL import Image
import matplotlib.pyplot as plt

engine = FashionSearchEngine()
engine.load_index('models/index')
print(f'Catalog size: {engine.catalog_size:,} items')

In [ ]:
# 6. Search by text
results = engine.search_by_text('blue denim jacket', top_k=6)

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for ax, r in zip(axes, results):
    img = Image.open(r['image_path'])
    ax.imshow(img)
    ax.set_title(f"{r['score']*100:.1f}%\n{r['category']}", fontsize=9)
    ax.axis('off')
plt.suptitle('Query: "blue denim jacket"', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Launch the Streamlit app (via localtunnel for Colab)
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time

proc = subprocess.Popen(['streamlit', 'run', 'app/app.py', '--server.port', '8501'])
time.sleep(4)
tunnel = ngrok.connect(8501)
print('ZaDeep Fashion is live at:', tunnel.public_url)